In [1]:
#
import requests
import pandas as pd 
pd.set_option('display.max_rows', None)
from IPython.display import HTML

#
import mysql.connector
from mysql.connector import Error

#
import numpy as np

#
import os 
from dotenv import load_dotenv
load_dotenv()
password_sql = os.getenv("PASS_SQL")

import json

# EXTRACCIÓN DE DATOS EN DEEZER Y LAST.FM

In [2]:
# Usamos la función para extraer todos los cantantes a un CSV 
def extraccion_deezer(endpoint):

    try:
        # 
        
        datos_musica = requests.get(endpoint)
        if datos_musica.status_code == 200:
            print ("API connected")
            # 
            df_musica = pd.DataFrame([datos_musica.json()])
            return df_musica
        else:
            # 
            print ("API failed")

    #         
    except requests.exceptions.ConnectionError as CnxE:
        print (CnxE)

    # 
    except requests.exceptions.Timeout as TO:
        print (TO)
    
    # 
    except requests.exceptions.RequestException as e:
        print (e) 

In [3]:
# Añadimos a una lista los ID de los cantantes o grupos que queremos extraer
artist_id = [12246, 160, 145, 564, 75491, 75798, 290, 483, 10803980, 1538640, 892, 10583405, 412, 13, 4050205, 384236, 119, 5620251, 259, 5962948, 196, 485, 425, 315929, 2446, 1755, 180, 98, 10977, 1434]

def extraer_artistas(artist_id):

    lista_artistas = []
    for artist in artist_id:
        endpoint = f"https://api.deezer.com/artist/{artist}"

        df_artistas = extraccion_deezer(endpoint)
        df_artistas = df_artistas[["id", "name"]]
        lista_artistas.append(df_artistas)
    df_final = pd.concat(lista_artistas)
    return df_final

In [4]:
df_artistas = extraer_artistas(artist_id)

API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected


In [5]:
HTML(df_artistas.to_html(index=False))

id,name
12246,Taylor Swift
160,Shakira
145,Beyoncé
564,Rihanna
75491,Lady Gaga
75798,Adele
290,Madonna
483,Britney Spears
10803980,BLACKPINK
1538640,Little Mix


In [6]:
##ESTE HAY QUE MODIFICARLO

def extraer_canciones(artist_id):

    lista_canciones_totales = []
    for artist in artist_id:
        endpoint = f"https://api.deezer.com/artist/{artist}/top?limit=50"
        canciones = extraccion_deezer(endpoint)
        lista_canciones_totales.extend(canciones["data"][0])
        
    lista_canciones = []
    for cancion in lista_canciones_totales:
        id_cancion = cancion["id"]
        id_artista = cancion["artist"]["id"]
        id_album = cancion["album"]["id"]
        titulo = cancion["title"]
        duracion = cancion["duration"]
        rank = cancion["rank"]
        if len(cancion["contributors"]) == 1:
            colaboradores = False
        else:
            colaboradores = True


        diccionario_cancion = {
                "id_cancion": id_cancion,
                "id_artista": id_artista,
                "id_album": id_album,
                "title": titulo,
                "duration": duracion,
                "rank": rank,
                "contributors_bool": colaboradores
            }         
                        
        #aqui hay qye nmeter lo de data, y ver que columnas queremos sacar
        lista_canciones.append(diccionario_cancion)
        
    df_final = pd.DataFrame(lista_canciones)
    return df_final, lista_canciones

In [7]:
df_canciones ,  mi_lista = extraer_canciones(artist_id)  ## IDEA SERIA QUE CADA
df_canciones.to_csv("listado_de_canciones.csv", index=False)



API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected


In [8]:
def conteo_artistas(artist_id):

    diccionario_artistas = {}
    for artist in artist_id:
        response = requests.get(f"https://api.deezer.com/artist/{artist}/top?limit=50")
        diccionario_artistas[artist] = len(response.json()["data"])
    return diccionario_artistas

In [9]:
diccionario_artistas = conteo_artistas(artist_id)
diccionario_artistas

{12246: 50,
 160: 50,
 145: 50,
 564: 50,
 75491: 50,
 75798: 50,
 290: 50,
 483: 50,
 10803980: 50,
 1538640: 50,
 892: 50,
 10583405: 50,
 412: 50,
 13: 50,
 4050205: 50,
 384236: 50,
 119: 50,
 5620251: 50,
 259: 50,
 5962948: 50,
 196: 50,
 485: 50,
 425: 50,
 315929: 50,
 2446: 50,
 1755: 50,
 180: 50,
 98: 50,
 10977: 50,
 1434: 50}

In [10]:
endpoint_genero = "https://api.deezer.com/genre/"  ## ¿meter dentro?

def extraer_genero(endpoint_genero):

    try:
        df_genero = extraccion_deezer(endpoint_genero)
        df_data = pd.DataFrame(df_genero["data"][0])
        df_final = df_data[["id","name"]]                       
        return df_final
    except:
        print ("error")

In [11]:
df_genero = extraer_genero(endpoint_genero)
HTML(df_genero.to_html(index=False))

API connected


id,name
0,Todos
132,Pop
116,Rap/Hip Hop
122,Reggaeton
152,Rock
113,Dance
165,R&B
85,Alternativo
106,Electro
466,Folk


In [12]:
def extraer_album():
    album_ids_unicos = set()

    for cancion in mi_lista:
        album_ids_unicos.add(cancion["id_album"])

    lista_albumes = []
    for album_id in album_ids_unicos:
        respuesta = requests.get(f"https://api.deezer.com/album/{album_id}")
        
        id_album = respuesta.json()["id"]
        id_artista = respuesta.json()["artist"]["id"]
        id_genero = respuesta.json()["genre_id"]
        titulo = respuesta.json()["title"]
        n_canciones = respuesta.json()["nb_tracks"]
        fecha_lanzamiento = respuesta.json()["release_date"]

        diccionario_albumes = {
                "id_album": id_album,
                "id_artista": id_artista,
                "id_genre": id_genero,
                "titulo": titulo,
                "n_canciones": n_canciones,
                "fecha_lanzamiento": fecha_lanzamiento
            }     

        
        lista_albumes.append(diccionario_albumes)
    
    df_final = pd.DataFrame(lista_albumes)
    return df_final
        


In [13]:
df_albumes = extraer_album()
df_albumes.to_csv("listado_de_albumes.csv", index=False)

In [14]:
def extraer_last_fm():

    lista_artistas = []
    df_artistas["name"].str.replace(" ", "+")
    for name in df_artistas["name"]:
        endpoint = f"https://ws.audioscrobbler.com/2.0/?method=artist.getInfo&artist={name}&api_key=68eddfdc6b072ad56527a4199d979038&format=json"
        df_last_fm = extraccion_deezer(endpoint)
        df_last_fm_datos = df_last_fm["artist"]
    
    
        for artista in df_last_fm_datos:
            nombre = artista["name"]
            popularidad = artista["stats"]["listeners"]
            reproducciones = artista["stats"]["playcount"]
            biografia = artista["bio"]["summary"]
            if len(artista["similar"]["artist"]) == 0:
                similares = False
            else:
                similares = True
            
            diccionario_artistas = {
                    "nombre": nombre,
                    "popularidad": popularidad,
                    "reproducciones": reproducciones,
                    "biografia": biografia,
                    "similares": similares
                }      
            
            lista_artistas.append(diccionario_artistas)

    df_final = pd.DataFrame(lista_artistas)
    df_final["biografia"] = df_final["biografia"].str.replace("\n", " ")
    return df_final, lista_artistas
        

    

In [15]:
df_artistas_fm, lista_artistas_fm = extraer_last_fm()

API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected


In [16]:
df_artistas_fm.to_csv("listado_de_artistas_fm.csv", index=False)

# INSERCIÓN EN MYSQL

In [17]:
def conectar_mysql(host="127.0.0.1", user="root", password=password_sql, database=None):
    try:
        cnx = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database        # si no le pasas base de datos, se conecta al servidor solo
        )
        print("Conexión exitosa")
        return cnx                      # con este return guarda la coneccion y puedo usar esta conexion despues 
    except Error as e:
        print(f"Error al conectar: {e}")

In [18]:
conexion = conectar_mysql()

Conexión exitosa


In [23]:
nombre_bd = "proyecto_music_stream_team1"

In [24]:
# 
def crear_basededatos(nombre_bd):
   
    try:
        # 
        with conexion.cursor() as cursor:
            query = f"CREATE DATABASE IF NOT EXISTS {nombre_bd}"
            # 
            cursor.execute(query)
            print ("Query exitosa")
 
    # 
    except Error as e:
        print (f"Error creando base de datos: {e}")

In [25]:
crear_basededatos(nombre_bd)

Query exitosa


In [26]:
# para crear tablas
def crear_tablas_genericas(nombre_bd, nombre_tabla, tabla_esquema):
   
    try:
        # 
        with conexion.cursor() as cursor:
            cursor.execute(f"USE {nombre_bd};")
            # 
            query = f''' CREATE TABLE IF NOT EXISTS {nombre_tabla} ({tabla_esquema});'''
            #
            cursor.execute(query)
            print ("Query creación exitosa")
   
    #
    except Error as e:
        print (f"Error creando tabla: {e}")

In [52]:
tabla_artista = 'artista'
tabla_genero_musical = 'genero_musical'
tabla_canciones = 'canciones'
tabla_album = 'album'

In [29]:
esquema_artista = '''id_artista INT PRIMARY KEY,
   nombre VARCHAR(30) NOT NULL,
   oyentes INT,
   reproducciones INT,
   biografia VARCHAR(1000) NOT NULL,
   genero VARCHAR(10) NOT NULL
   '''

In [53]:
esquema_genero_musical = '''id_genero INT PRIMARY KEY, 
nombre VARCHAR(40)'''

In [32]:
esquema_canciones = '''id_cancion INT PRIMARY KEY,
id_artista INT NOT NULL,
id_album INT NOT NULL,
titulo VARCHAR(200) NOT NULL,
duracion INT,
ranking_lista INT,
colaboraciones BOOLEAN
'''

In [48]:
esquema_album = ''' id_album INT PRIMARY KEY,
id_artista INT,
id_genero INT,
titulo VARCHAR(200) NOT NULL,
numero_canciones INT,
fecha_lanzamiento DATE
'''

In [34]:
crear_tablas_genericas(nombre_bd,tabla_artista, esquema_artista)

Query creación exitosa


In [54]:
crear_tablas_genericas(nombre_bd, tabla_genero_musical, esquema_genero_musical)

Query creación exitosa


In [36]:
crear_tablas_genericas(nombre_bd,tabla_canciones, esquema_canciones)

Query creación exitosa


In [49]:
crear_tablas_genericas(nombre_bd,tabla_album, esquema_album)

Query creación exitosa
